# classical machine-learning baselines

This notebook implements **classical machine-learning baselines** for BCE prediction using precomputed CSV feature files.

## What this notebook does
- loads feature matrices for multiple feature sets (for example `F1`, `F2` and `PSTPP`)
- defines baseline ML models and hyperparameter search spaces
- tunes each model on the **training split only**
- performs **fold-wise evaluation** on the training set using the selected best hyperparameters
- trains each tuned model on the full training split
- evaluates on unseen splits (`test`, `ext1`, `ext2`)
- exports result tables for reporting and downstream comparison

## Important usage notes
- This notebook expects a folder structure like:
  `features_selected/<FeatureSet>/{train,test,ext1,ext2}.csv`
- Each CSV must contain a binary label column named `label` unless you update `LABEL_COL`.
- Comments were added to make the workflow easier to understand and reproduce.

## Suggested execution order
Run the notebook from top to bottom without skipping cells, because later cells depend on:
- configuration variables
- loaded data dictionaries
- tuned hyperparameters
- trained final models

### Classical ML Baselines for iDeepLBCE using CSV Feature Files

### 1. Setup and Libraries

In [ ]:
# Import core libraries, define paths, and create the results directory.
# Note: some imports may remain unused depending on which exact analysis path you run.
import os
import json
import warnings
import numpy as np
import pandas as pd

from copy import deepcopy
from collections import defaultdict

# Sklearn
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    confusion_matrix,
    brier_score_loss
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
# Stats
from scipy.stats import wilcoxon, friedmanchisquare
from statsmodels.stats.contingency_tables import mcnemar

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
LABEL_COL = "label"   # change this if your label column has another name
DATA_ROOT = "features_selected"  # change this if your data is in a different location
RESULTS_DIR = "results"

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Setup complete.")

### 2. Configuration

In [ ]:
# Configure the feature sets, evaluation splits, and ML models to benchmark.
# `SEARCH_TYPE` controls whether hyperparameter tuning uses grid search or randomized search.
FEATURE_SETS = ["F2", "F1", "PSTPP"]

SPLITS = ["train", "test", "ext1", "ext2"]

# You can reduce or expand this list
MODEL_NAMES = ["LR", "RF", "ET", "XGB"]
# Search type: choose "grid" or "random"
SEARCH_TYPE = "random"

# Only used if SEARCH_TYPE == "random"
N_ITER_RANDOM = 20

# Tuning metric
TUNING_SCORING = "accuracy" # change this to your preferred metric for hyperparameter tuning

print("Feature sets:", FEATURE_SETS)
print("Models:", MODEL_NAMES)
print("Search type:", SEARCH_TYPE)
print("Scoring:", TUNING_SCORING)

### 3. Load CSV data

In [ ]:
# Load each feature-set CSV split into memory.
# The notebook expects a binary label column (`label`) and uses all remaining columns as features.
def load_csv_split(feature_name, split_name, root=DATA_ROOT, label_col=LABEL_COL):
    csv_path = os.path.join(root, feature_name, f"{split_name}.csv")
    df = pd.read_csv(csv_path)

    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found in {csv_path}")

    X = df.drop(columns=[label_col]).values
    y = df[label_col].values
    feature_cols = [c for c in df.columns if c != label_col]

    return X, y, feature_cols, df


def load_feature_set_csv(feature_name, root=DATA_ROOT, label_col=LABEL_COL):
    data = {}
    for split in SPLITS:
        X, y, feat_cols, df = load_csv_split(feature_name, split, root=root, label_col=label_col)
        data[f"X_{split}"] = X
        data[f"y_{split}"] = y
        data[f"df_{split}"] = df
        data["feature_cols"] = feat_cols
    return data


data_dict = {}
for fs in FEATURE_SETS:
    data_dict[fs] = load_feature_set_csv(fs)

for fs, d in data_dict.items():
    print(f"\nFeature set: {fs}")
    for split in SPLITS:
        print(f"  {split}: X = {d[f'X_{split}'].shape}, y = {d[f'y_{split}'].shape}")

### 4. Metric functions

In [ ]:
# Define reusable binary-classification metrics.
# `evaluate_binary_classifier` takes probabilities, applies a threshold, and returns a metrics dictionary.

def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape != (2, 2):
        raise ValueError("Specificity requires binary labels with 2 classes.")
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0


def evaluate_binary_classifier(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "ACC": accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob),
        "Sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Brier": brier_score_loss(y_true, y_prob)
    }
    return metrics

### 5. Classical ML models and hyperparameter grids

In [ ]:
# Define the baseline ML models and their hyperparameter spaces.
# Pipelines currently include a MinMaxScaler followed by the classifier.
# You can add or remove models here without changing the downstream logic.

def get_model_configs():
    configs = {}

    # Logistic Regression
    configs["LR"] = {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
        ]),
        "param_grid": {
            "clf__C": [0.01, 0.1, 1, 10, 100],
            "clf__solver": ["lbfgs", "liblinear"],
            "clf__penalty": ["l2"]
        }
    }

    # Random Forest
    configs["RF"] = {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", RandomForestClassifier(random_state=RANDOM_STATE,n_jobs=-1))
        ]),
        "param_grid": {
            "clf__n_estimators": [200, 500],
            "clf__max_depth": [3, 6, 10],
            "clf__min_samples_split": [2, 5, 10],
            "clf__min_samples_leaf": [1, 2, 4],
            "clf__max_features": ["sqrt", "log2"]
        }
    }

    # Extra Trees
    configs["ET"] = {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", ExtraTreesClassifier(random_state=RANDOM_STATE,n_jobs=-1))
        ]),
        "param_grid": {
            "clf__n_estimators": [200, 500],
            "clf__max_depth": [3, 6, 10],
            "clf__min_samples_split": [2, 5, 10],
            "clf__min_samples_leaf": [1, 2, 4],
            "clf__max_features": ["sqrt", "log2"]
        }
    }

    # XGBoost
    configs["XGB"] = {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1))
        ]),
        "param_grid": {
            "clf__n_estimators": [200, 500],
            "clf__max_depth": [3, 6, 10],
            "clf__learning_rate": [0.01, 0.1, 0.2],
            "clf__subsample": [0.8, 1.0],
            "clf__colsample_bytree": [0.8, 1.0]
        }
    }

    return configs

MODEL_CONFIGS = get_model_configs()
MODEL_CONFIGS.keys()

### 6. Nested CV for fair tuning and evaluation

In [ ]:
# Tune one model on the training set only using 5-fold stratified CV.
# This keeps model selection separate from the unseen test/ext1/ext2 evaluation.

def tune_model_once(X_train, y_train, model_name, model_config,
                    scoring=TUNING_SCORING,
                    search_type=SEARCH_TYPE,
                    n_iter=N_ITER_RANDOM):
    """
    Tune hyperparameters once using 5-fold CV on training data only.
    Returns best hyperparameters and best CV score.
    """
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    if search_type == "grid":
        search = GridSearchCV(
            estimator=clone(model_config["pipeline"]),
            param_grid=model_config["param_grid"],
            cv=cv,
            scoring=scoring,
            refit=False,
            n_jobs=-1
        )
    elif search_type == "random":
        search = RandomizedSearchCV(
            estimator=clone(model_config["pipeline"]),
            param_distributions=model_config["param_grid"],
            n_iter=n_iter,
            cv=cv,
            scoring=scoring,
            refit=False,
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    else:
        raise ValueError("search_type must be either 'grid' or 'random'.")

    search.fit(X_train, y_train)

    return search.best_params_, search.best_score_

### 7. Train final model using the selected best hyperparameters

No re-tuning here.

In [ ]:
# Fit a final version of the selected model using the chosen best hyperparameters on the full training data.

def train_final_model_from_best_params(X_train, y_train, model_config, best_params):
    """
    Train final model on the full training set using already selected best params.
    """
    final_model = clone(model_config["pipeline"])
    final_model.set_params(**best_params)
    final_model.fit(X_train, y_train)
    return final_model

### 8. Tune all models once and save best hyperparameters

### 8. 1. Add a new function for fold-wise evaluation using fixed best parameters

In [ ]:
# After hyperparameter tuning, this helper runs a manual 5-fold CV with the *fixed* best parameters.
# This produces fold-wise metrics that can later be summarized as mean ± SD.
def get_foldwise_results_with_best_params(X_train, y_train, model_name, model_config, best_params):
    """
    After tuning is complete, evaluate the model with fixed best hyperparameters
    in a manual 5-fold CV loop and store all fold-wise metrics.
    """
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    fold_rows = []

    for fold_idx, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        model = clone(model_config["pipeline"])
        model.set_params(**best_params)
        model.fit(X_tr, y_tr)

        y_prob = model.predict_proba(X_val)[:, 1]
        metrics = evaluate_binary_classifier(y_val, y_prob)

        row = {
            "fold": fold_idx,
            "Model": model_name,
            "BestParams": json.dumps(best_params)
        }
        row.update(metrics)
        fold_rows.append(row)

    return pd.DataFrame(fold_rows)

### 8.2. Tuning loop to also collect fold-wise results

In [ ]:
# Main tuning loop.
# For each feature set and each classical model:
# 1) tune hyperparameters on the training split
# 2) store the best parameters and best CV score
# 3) collect fold-wise metrics using the tuned parameter set
best_params_store = defaultdict(dict)
tuning_rows = []
foldwise_results_all = []

for feature_name in FEATURE_SETS:
    X_train = data_dict[feature_name]["X_train"]
    y_train = data_dict[feature_name]["y_train"]

    print(f"\n========================")
    print(f"Tuning on training set: {feature_name}")
    print(f"========================")

    for model_name in MODEL_NAMES:
        print(f"Tuning {model_name} ...")

        best_params, best_cv_score = tune_model_once(
            X_train=X_train,
            y_train=y_train,
            model_name=model_name,
            model_config=MODEL_CONFIGS[model_name],
            scoring=TUNING_SCORING,
            search_type=SEARCH_TYPE,
            n_iter=N_ITER_RANDOM
        )

        best_params_store[feature_name][model_name] = best_params

        tuning_rows.append({
            "FeatureSet": feature_name,
            "Model": model_name,
            "BestCVScore": best_cv_score,
            "Scoring": TUNING_SCORING,
            "SearchType": SEARCH_TYPE,
            "BestParams": json.dumps(best_params)
        })

        print(f"  Best CV {TUNING_SCORING}: {best_cv_score:.4f}")
        print(f"  Best params: {best_params}")

        # NEW: collect fold-wise full metrics using fixed best params
        df_foldwise = get_foldwise_results_with_best_params(
            X_train=X_train,
            y_train=y_train,
            model_name=model_name,
            model_config=MODEL_CONFIGS[model_name],
            best_params=best_params
        )
        df_foldwise["FeatureSet"] = feature_name
        foldwise_results_all.append(df_foldwise)

tuning_results_df = pd.DataFrame(tuning_rows)
tuning_results_df.to_csv(os.path.join(RESULTS_DIR, "tuning_summary.csv"), index=False)

foldwise_results_df = pd.concat(foldwise_results_all, ignore_index=True)
foldwise_results_df.to_csv(os.path.join(RESULTS_DIR, "foldwise_cv_results.csv"), index=False)

print("\nSaved:", os.path.join(RESULTS_DIR, "tuning_summary.csv"))
print("Saved:", os.path.join(RESULTS_DIR, "foldwise_cv_results.csv"))

tuning_results_df

In [ ]:
# Build a mean ± SD summary table from the stored fold-wise CV results.
# This is useful for manuscript tables and reporting training-set stability.
metric_cols = ["ACC", "F1", "MCC", "ROC_AUC", "PR_AUC", "Sensitivity", "Specificity", "Precision", "Brier"]

summary_rows = []

for (feature_name, model_name), grp in foldwise_results_df.groupby(["FeatureSet", "Model"]):
    row = {
        "FeatureSet": feature_name,
        "Model": model_name
    }
    for m in metric_cols:
        row[f"{m}_mean"] = grp[m].mean()
        row[f"{m}_sd"] = grp[m].std(ddof=1)
        row[f"{m}_mean_sd"] = f"{grp[m].mean():.4f} ± {grp[m].std(ddof=1):.4f}"
    summary_rows.append(row)

cv_metrics_summary_df = pd.DataFrame(summary_rows)
cv_metrics_summary_df.to_csv(os.path.join(RESULTS_DIR, "cv_metrics_summary_mean_sd.csv"), index=False)

print("Saved:", os.path.join(RESULTS_DIR, "cv_metrics_summary_mean_sd.csv"))
cv_metrics_summary_df

### 8.3. Display fold-wise results

In [ ]:
# Quick preview of the fold-wise results dataframe.
# Useful for checking whether all feature sets / models were evaluated correctly.
foldwise_results_df.head(20)

### 8.4. Create mean ± SD summary from fold-wise results

In [ ]:
# Duplicate summary-generation block retained from the original notebook.
# It reproduces the same mean ± SD summary export without changing logic.
metric_cols = ["ACC", "F1", "MCC", "ROC_AUC", "PR_AUC", "Sensitivity", "Specificity", "Precision", "Brier"]

summary_rows = []

for (feature_name, model_name), grp in foldwise_results_df.groupby(["FeatureSet", "Model"]):
    row = {
        "FeatureSet": feature_name,
        "Model": model_name
    }
    for m in metric_cols:
        row[f"{m}_mean"] = grp[m].mean()
        row[f"{m}_sd"] = grp[m].std(ddof=1)
        row[f"{m}_mean_sd"] = f"{grp[m].mean():.4f} ± {grp[m].std(ddof=1):.4f}"
    summary_rows.append(row)

cv_metrics_summary_df = pd.DataFrame(summary_rows)
cv_metrics_summary_df.to_csv(os.path.join(RESULTS_DIR, "cv_metrics_summary_mean_sd.csv"), index=False)

print("Saved:", os.path.join(RESULTS_DIR, "cv_metrics_summary_mean_sd.csv"))
cv_metrics_summary_df

### 9. Fit final tuned models on full training set

In [ ]:
# Train final tuned models on the full training split.
# Then evaluate each trained model on unseen splits: `test`, `ext1`, and `ext2`.

trained_models = defaultdict(dict)
final_rows = []

for feature_name in FEATURE_SETS:
    X_train = data_dict[feature_name]["X_train"]
    y_train = data_dict[feature_name]["y_train"]

    print(f"\n========================")
    print(f"Final training for: {feature_name}")
    print(f"========================")

    for model_name in MODEL_NAMES:
        best_params = best_params_store[feature_name][model_name]

        final_model = train_final_model_from_best_params(
            X_train=X_train,
            y_train=y_train,
            model_config=MODEL_CONFIGS[model_name],
            best_params=best_params
        )

        trained_models[feature_name][model_name] = final_model

        for split in ["test", "ext1", "ext2"]:
            X_eval = data_dict[feature_name][f"X_{split}"]
            y_eval = data_dict[feature_name][f"y_{split}"]

            y_prob = final_model.predict_proba(X_eval)[:, 1]
            metrics = evaluate_binary_classifier(y_eval, y_prob)

            row = {
                "FeatureSet": feature_name,
                "Model": model_name,
                "Split": split,
                "BestParams": json.dumps(best_params)
            }
            row.update(metrics)
            final_rows.append(row)

            print(f"{model_name} | {split} | ROC_AUC={metrics['ROC_AUC']:.4f} | ACC={metrics['ACC']:.4f}")

final_results_df = pd.DataFrame(final_rows)
final_results_df.to_csv(os.path.join(RESULTS_DIR, "final_unseen_results.csv"), index=False)

print("\nSaved:", os.path.join(RESULTS_DIR, "final_unseen_results.csv"))
final_results_df.head()

### 10. Format the table style

In [ ]:
# Format the main unseen-results table for easier reading and export.
# Values are rendered to four decimal places for reporting.
# Cell 10
metric_cols = ["ACC", "F1", "MCC", "ROC_AUC", "PR_AUC", "Sensitivity", "Specificity", "Precision", "Brier"]

formatted_final_df = final_results_df.copy()
for col in metric_cols:
    formatted_final_df[col] = formatted_final_df[col].map(lambda x: f"{x:.4f}")

formatted_final_df = formatted_final_df.sort_values(["Split", "FeatureSet", "ROC_AUC"], ascending=[True, True, False])
formatted_final_df.to_csv(os.path.join(RESULTS_DIR, "final_unseen_results_formatted.csv"), index=False)

print("Saved:", os.path.join(RESULTS_DIR, "final_unseen_results_formatted.csv"))
formatted_final_df

### 11. Select the best classical model per feature set and split

In [ ]:
# Select the best classical model per feature set and split.
# The current selection rule uses the highest ROC-AUC.
# Cell 11
best_classical_rows = []

for (feature_name, split), grp in final_results_df.groupby(["FeatureSet", "Split"]):
    best_row = grp.sort_values("ROC_AUC", ascending=False).iloc[0]

    best_classical_rows.append({
        "FeatureSet": feature_name,
        "Split": split,
        "BestModel": best_row["Model"],
        "ACC": best_row["ACC"],
        "F1": best_row["F1"],
        "MCC": best_row["MCC"],
        "ROC_AUC": best_row["ROC_AUC"],
        "PR_AUC": best_row["PR_AUC"],
        "Sensitivity": best_row["Sensitivity"],
        "Specificity": best_row["Specificity"],
        "Precision": best_row["Precision"],
        "Brier": best_row["Brier"]
    })

best_classical_df = pd.DataFrame(best_classical_rows)
best_classical_df.to_csv(os.path.join(RESULTS_DIR, "best_classical_per_split.csv"), index=False)

print("Saved:", os.path.join(RESULTS_DIR, "best_classical_per_split.csv"))
best_classical_df

In [ ]:
# Reload the exported unseen-results CSV if you want to inspect it independently of the in-memory dataframe.
classical_df = pd.read_csv("results/final_unseen_results.csv")

### 14. Export compact summary tables

In [ ]:
# Export compact summary tables that can be used directly in downstream analysis or manuscript preparation.
# Cell 16
tuning_results_df.to_csv(os.path.join(RESULTS_DIR, "table_tuning_summary.csv"), index=False)
formatted_final_df.to_csv(os.path.join(RESULTS_DIR, "table_unseen_results.csv"), index=False)
best_classical_df.to_csv(os.path.join(RESULTS_DIR, "table_best_classical_summary.csv"), index=False)

print("Saved:")
print("-", os.path.join(RESULTS_DIR, "table_tuning_summary.csv"))
print("-", os.path.join(RESULTS_DIR, "table_unseen_results.csv"))
print("-", os.path.join(RESULTS_DIR, "table_best_classical_summary.csv"))

## Final note

If you adapt this notebook for a new project:
1. update `DATA_ROOT`
2. verify the CSV split names
3. confirm the label column name
4. check whether the hyperparameter grids remain appropriate for your dataset size

The current notebook is especially useful as a **transparent classical-baseline benchmark** against the deep-learning models.